# Trabajo Práctico Integrador – Entrega 2

**Materia:** Introducción al Análisis de Datos  
**Tema:** Cancelación de reservas hoteleras  
**Comisión:** 10 (Grupo J)  
**Integrante:** Viruel, Nicolás  
**Entrega:** 2 – Unidad N° 2  
**Dataset:** `hotel_booking_TPI_grupo_J.csv`  
**Variable objetivo:** `is_canceled`  
**Año:** 2026

> Notebook independiente: ejecutable desde cero.

## 1. Contexto

Analizamos reservas hoteleras para estudiar cancelaciones. En esta entrega aplicamos **diagnóstico, limpieza y transformación** del dataset asignado a la comisión 10 (Grupo J).

## 2. Carga reproducible del dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../datos')
archivo = DATA_DIR / 'hotel_booking_TPI_grupo_J.csv'
df_hotel = pd.read_csv(archivo)
print('Dataset cargado:', df_hotel.shape)

Dataset cargado: (25000, 32)


## 3. Copia de trabajo y bitácora

In [2]:
df_hotel_original = df_hotel.copy()
df = df_hotel.copy()

registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        'problema_detectado': problema,
        'variable': variable,
        'decision': decision,
        'justificacion': justificacion,
    })

## 4. Diagnóstico de calidad

In [3]:
print('Dimensiones:', df.shape)
print('Duplicados exactos:', df.duplicated().sum())
df.info()

Dimensiones: (25000, 32)
Duplicados exactos: 0
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                           25000 non-null  str    
 2   is_canceled                     25000 non-null  int64  
 3   lead_time                       25000 non-null  int64  
 4   arrival_date_year               25000 non-null  int64  
 5   arrival_date_month              25000 non-null  str    
 6   arrival_date_week_number        25000 non-null  int64  
 7   arrival_date_day_of_month       25000 non-null  int64  
 8   arrival_date                    25000 non-null  str    
 9   stays_in_weekend_nights         25000 non-null  int64  
 10  stays_in_week_nights            25000 non-null  int64  
 11  adults                          25000 non-null  int64  
 

In [4]:
cols_clave = [
    'children', 'adults', 'babies', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adr', 'lead_time', 'country', 'agent', 'company',
    'hotel', 'market_segment', 'deposit_type', 'customer_type', 'is_canceled',
]
df[cols_clave].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
children,25000.0,NaN,NaN,NaN,0.10308,0.393218,0.0,0.0,0.0,0.0,3.0
adults,25000.0,NaN,NaN,NaN,1.85444,0.571371,0.0,2.0,2.0,2.0,40.0
babies,25000.0,NaN,NaN,NaN,0.00804,0.091519,0.0,0.0,0.0,0.0,2.0
stays_in_weekend_nights,25000.0,NaN,NaN,NaN,0.92044,0.998193,0.0,0.0,1.0,2.0,18.0
stays_in_week_nights,25000.0,NaN,NaN,NaN,2.499,1.903613,0.0,1.0,2.0,3.0,42.0
adr,25000.0,NaN,NaN,NaN,101.681678,47.969113,0.0,69.0275,94.5,126.0,387.0
lead_time,25000.0,NaN,NaN,NaN,104.22324,107.098048,0.0,18.0,69.0,160.0,629.0
country,24883,134,PRT,10159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agent,21521.0,NaN,NaN,NaN,86.426746,111.048259,1.0,9.0,14.0,229.0,535.0
company,1440.0,NaN,NaN,NaN,187.325694,128.592996,9.0,62.0,186.0,268.0,541.0


In [5]:
nulos = df.isna().sum().sort_values(ascending=False)
print(nulos[nulos > 0])
registrar('Alta proporción de company nulo', 'company', 'Conservar NaN',
          'Patrón real de reservas sin empresa (~94% nulo).')
registrar('Agent nulo en ~14%', 'agent', 'Conservar NaN',
          'Reservas directas o sin intermediario.')
registrar('Country ausente', 'country', 'Conservar NaN',
          'Pocas filas; eliminar sesgaría países minoritarios.')

company    23560
agent       3479
country      117
dtype: int64


In [6]:
print('adr <= 0:', (df['adr'] <= 0).sum())
print('adults == 0:', (df['adults'] == 0).sum())
print('Sin huéspedes:', ((df['adults'] + df['children'].fillna(0) + df['babies']) == 0).sum())
registrar('Tarifa adr <= 0', 'adr', 'Conservar y marcar',
          'Puede ser cortesía o error; no se elimina sin validación de negocio.')
registrar('lead_time elevado', 'lead_time', 'Conservar',
          'Anticipación extrema es posible en resorts.')

adr <= 0: 446
adults == 0: 80
Sin huéspedes: 37


In [7]:
def detectar_outliers_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lim_inf) | (s > lim_sup)]

for col in ['lead_time', 'adr']:
    out = detectar_outliers_iqr(df[col])
    print(f'{col}: {len(out)} atípicos IQR')

lead_time: 657 atípicos IQR


adr: 777 atípicos IQR


## 5. Limpieza aplicada

In [8]:
df['children'] = df['children'].fillna(0)
registrar('children nulo', 'children', 'Imputar 0',
          'Sin hijos registrados se interpreta como 0 huéspedes menores.')

df['country'] = df['country'].replace('', np.nan)
df['arrival_date'] = pd.to_datetime(df['arrival_date'], errors='coerce')
registrar('Unificación temporal', 'arrival_date', 'Convertir a datetime',
          'Habilita variables derivadas de calendario.')

## 6. Transformación de variables

In [9]:
# Variables derivadas sugeridas
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['estimated_stay_amount'] = df['adr'] * df['total_nights']
df['family_booking'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)
registrar('Duración de estadía dispersa', 'stays_in_weekend_nights, stays_in_week_nights',
          'Crear total_nights', 'Resume noches totales para importe y segmentación.')
registrar('Huéspedes en varias columnas', 'adults, children, babies',
          'Crear total_guests', 'Variable única para capacidad/ocupación.')
registrar('Importe no explícito', 'adr, total_nights',
          'estimated_stay_amount = adr * total_nights',
          'Aproxima ingreso de la reserva para análisis económico.')
registrar('Reservas familiares', 'children, babies',
          'family_booking binaria', 'Marca presencia de menores.')
df[['total_nights', 'total_guests', 'estimated_stay_amount', 'family_booking']].head()

,total_nights,total_guests,estimated_stay_amount,family_booking
0,7,2.0,300.65,0
1,3,2.0,192.00,0
2,2,2.0,203.00,0
3,2,1.0,228.00,0
4,2,2.0,178.20,0


In [10]:
# Discretización lead_time
df['lead_time_segment'] = pd.cut(
    df['lead_time'],
    bins=[-1, 7, 30, 90, df['lead_time'].max()],
    labels=['0-7 días', '8-30 días', '31-90 días', '91+ días'],
)
registrar('lead_time continuo amplio', 'lead_time',
          'Discretizar en lead_time_segment',
          'Facilita comparar cancelaciones por anticipación de reserva.')
df['lead_time_segment'].value_counts()

lead_time_segment
91+ días      10712
31-90 días     6192
0-7 días       4151
8-30 días      3945
Name: count, dtype: int64

In [11]:
# Variables temporales
df['arrival_month'] = df['arrival_date'].dt.month
df['arrival_dayofweek'] = df['arrival_date'].dt.dayofweek
registrar('Fecha de llegada', 'arrival_date',
          'Extraer arrival_month y arrival_dayofweek',
          'Captura estacionalidad y día de la semana.')

In [12]:
# Codificación categórica (one-hot de hotel y deposit_type)
hotel_dummies = pd.get_dummies(df['hotel'], prefix='hotel', drop_first=True)
deposit_dummies = pd.get_dummies(df['deposit_type'], prefix='deposit', drop_first=True)
df = pd.concat([df, hotel_dummies, deposit_dummies], axis=1)
registrar('hotel nominal', 'hotel', 'One-Hot con drop_first',
          'Evita multicolinealidad perfecta en modelos lineales.')
registrar('deposit_type ordinal/nominal', 'deposit_type', 'One-Hot',
          'Política de depósito puede explicar cancelaciones.')
list(hotel_dummies.columns) + list(deposit_dummies.columns)

['hotel_Resort Hotel', 'deposit_Non Refund', 'deposit_Refundable']

In [13]:
# Normalización Min-Max de adr (solo valores positivos para referencia)
adr_pos = df.loc[df['adr'] > 0, 'adr']
adr_min, adr_max = adr_pos.min(), adr_pos.max()
df['adr_minmax'] = np.where(
    df['adr'] > 0,
    (df['adr'] - adr_min) / (adr_max - adr_min),
    np.nan,
)
registrar('adr en escala amplia', 'adr', 'adr_minmax en [0,1] para adr>0',
          'Escala comparable con otras variables normalizadas; adr<=0 queda NaN.')

In [14]:
# Estandarización de total_guests
tg_mean = df['total_guests'].mean()
tg_std = df['total_guests'].std()
df['total_guests_standard'] = (df['total_guests'] - tg_mean) / tg_std
registrar('total_guests para modelos', 'total_guests',
          'Estandarizar → total_guests_standard',
          'Centra y escala huéspedes totales para algoritmos sensibles a magnitud.')

## 7. Bitácora del proceso

In [15]:
bitacora = pd.DataFrame(registros_bitacora)
bitacora

,problema_detectado,variable,decision,justificacion
0,Alta proporción de company nulo,company,Conservar NaN,Patrón real de reservas sin empresa (~94% nulo).
1,Agent nulo en ~14%,agent,Conservar NaN,Reservas directas o sin intermediario.
2,Country ausente,country,Conservar NaN,Pocas filas; eliminar sesgaría países minorita...
3,Tarifa adr <= 0,adr,Conservar y marcar,Puede ser cortesía o error; no se elimina sin ...
4,lead_time elevado,lead_time,Conservar,Anticipación extrema es posible en resorts.
5,children nulo,children,Imputar 0,Sin hijos registrados se interpreta como 0 hué...
6,Unificación temporal,arrival_date,Convertir a datetime,Habilita variables derivadas de calendario.
7,Duración de estadía dispersa,"stays_in_weekend_nights, stays_in_week_nights",Crear total_nights,Resume noches totales para importe y segmentac...
8,Huéspedes en varias columnas,"adults, children, babies",Crear total_guests,Variable única para capacidad/ocupación.
9,Importe no explícito,"adr, total_nights",estimated_stay_amount = adr * total_nights,Aproxima ingreso de la reserva para análisis e...


## 8. Dataset preparado

In [16]:
df_hotel_preparado = df.copy()
print('Original:', df_hotel_original.shape)
print('Preparado:', df_hotel_preparado.shape)
print('Columnas nuevas:', set(df_hotel_preparado.columns) - set(df_hotel_original.columns))
df_hotel_preparado[['is_canceled', 'total_nights', 'total_guests', 'estimated_stay_amount',
                     'lead_time_segment', 'family_booking', 'adr_minmax']].head(10)

Original: (25000, 32)
Preparado: (25000, 44)
Columnas nuevas: {'adr_minmax', 'deposit_Non Refund', 'hotel_Resort Hotel', 'family_booking', 'estimated_stay_amount', 'arrival_month', 'total_guests', 'total_guests_standard', 'total_nights', 'arrival_dayofweek', 'deposit_Refundable', 'lead_time_segment'}


,is_canceled,total_nights,total_guests,estimated_stay_amount,lead_time_segment,family_booking,adr_minmax
0,0,7,2.0,300.65,91+ días,0,0.108679
1,1,3,2.0,192.00,31-90 días,0,0.163212
2,1,2,2.0,203.00,91+ días,0,0.260363
3,0,2,1.0,228.00,31-90 días,0,0.292746
4,1,2,2.0,178.20,91+ días,0,0.228238
5,0,7,2.0,483.00,91+ días,0,0.176166
6,1,4,2.0,520.00,91+ días,0,0.334197
7,1,1,2.0,80.00,91+ días,0,0.204663
8,0,1,2.0,100.00,31-90 días,0,0.256477
9,0,4,1.0,216.00,31-90 días,0,0.137306


## 9. Cierre

El dataset quedó **diagnosticado, limpiado y transformado** con criterio analítico: se conservó el original, se documentaron faltantes e inconsistencias, se crearon variables derivadas (`total_nights`, `total_guests`, `estimated_stay_amount`, `family_booking`), segmentos de `lead_time`, codificación de categorías y escalado de variables numéricas. Este conjunto preparado habilita análisis exploratorio y modelado de `is_canceled` en entregas posteriores.